# Site04: B4 ellipsoid-normal cross-check with NAIF azlcpo

**Purpose:** Compare an independently assembled local-basis angle calculation against NAIF SPICE `azlcpo` for S04-P02 at three fixed epochs (00:00, 12:00, 23:00 UTC on 2025-01-01), for both Sun and Earth.

This notebook loads the CSV and kernel/run manifest from notebook 03, verifies the cached kernel SHA-256 values against that manifest, then constructs the observer and ellipsoid normal. It writes a cross-check CSV and JSON run manifest alongside the original artifacts.

NAIF defines the local topocentric +Z axis as the outward ellipsoid surface normal and +X as the projection of the body's +Z axis onto the tangent plane. The SPICE call below uses `azccw=False` for clockwise azimuth from north and `elplsz=True` for positive elevation toward local +Z. SPICE `azlcpo` expects observer coordinates in kilometres.

**B4 criterion:** maximum absolute azimuth and elevation difference between our ellipsoid-normal vector-to-local-basis conversion and `azlcpo` is less than 1e-6 degrees for all six target/epoch comparisons. This checks the local-angle conversion against NAIF's routine. It does not independently validate DE421 ephemeris truth, raster frame provenance, terrain visibility, or the radial-sphere results from notebook 03.

References:
- NAIF `azlcpo_c`: https://naif.jpl.nasa.gov/pub/naif/toolkit_docs/C/cspice/azlcpo_c.html
- NAIF `nearpt_c`: https://naif.jpl.nasa.gov/pub/naif/toolkit_docs/C/cspice/nearpt_c.html
- NAIF `surfnm_c`: https://naif.jpl.nasa.gov/pub/naif/toolkit_docs/C/cspice/surfnm_c.html

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import math
import time

import numpy as np
import pandas as pd
import spiceypy as spice

NOTEBOOK_RUN_T0 = time.perf_counter()
OUTPUT_DIR = Path.home() / "research-results" / "site04-spice-baseline"
CSV_PATH = OUTPUT_DIR / "S04-P02_2025-01-01_hourly_geometric_angles.csv"
BASELINE_MANIFEST_PATH = OUTPUT_DIR / "kernel_and_run_manifest.json"
CROSSCHECK_CSV_PATH = OUTPUT_DIR / "S04-P02_azlcpo_crosscheck.csv"
CROSSCHECK_MANIFEST_PATH = OUTPUT_DIR / "B4_validation_manifest.json"
KERNEL_DIR = Path.home() / ".cache" / "lunar-navigator" / "de421-kernels"
TOLERANCE_DEG = 1e-6

for path in (CSV_PATH, BASELINE_MANIFEST_PATH):
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}. Run notebook 03 first, then open this notebook from a writable copy."
        )

baseline = json.loads(BASELINE_MANIFEST_PATH.read_text(encoding="utf-8"))
baseline_df = pd.read_csv(CSV_PATH)
print("baseline_rows:", len(baseline_df))
print("output_dir:", OUTPUT_DIR)

## 1. Verify and load the exact kernel inputs from notebook 03

This cell refuses to use a cached kernel whose hash differs from the baseline manifest. It then loads the same named kernels so the two calculations share identical ephemeris and frame inputs.

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

kernel_manifest = baseline.get("kernel_manifest", [])
if len(kernel_manifest) != 5:
    raise ValueError(f"Expected five recorded kernels, found {len(kernel_manifest)}")

kernel_paths = []
for item in kernel_manifest:
    path = KERNEL_DIR / item["filename"]
    if not path.is_file():
        raise FileNotFoundError(f"Kernel missing from cache: {path}")
    actual_sha256 = sha256_file(path)
    if actual_sha256 != item["sha256"]:
        raise ValueError(
            f"Kernel hash mismatch for {item['filename']}: "
            f"baseline={item['sha256']} current={actual_sha256}"
        )
    kernel_paths.append(path)

spice.kclear()
for path in kernel_paths:
    spice.furnsh(str(path))

frame_id = spice.namfrm("MOON_ME")
if frame_id == 0:
    raise RuntimeError("MOON_ME did not resolve after loading the baseline kernels.")

print("MOON_ME frame ID:", frame_id)
print("verified kernel count:", len(kernel_paths))
print("all cached kernel SHA-256 values match notebook 03 manifest")

## 2. Rebuild observer point and ellipsoid local frame

The observer position is recreated from the product's reference sphere and DEM height. For the NAIF comparison, the local up vector is the surface normal at the nearest point on the Moon reference ellipsoid. The local north axis is the body's +Z axis projected onto the tangent plane; east is then formed to make the local basis right-handed.

The resulting radial-sphere and ellipsoid normals need not be identical. Their angle difference is reported separately from the B4 conversion error.

In [ ]:
point = baseline["point"]
longitude_deg = float(point["longitude_deg_from_raster_crs"])
latitude_deg = float(point["latitude_deg_from_raster_crs"])
radius_m = 1_737_400.0 + float(point["z_m_above_reference_sphere"])
observer_xyz_km = np.asarray(
    spice.latrec(
        radius_m / 1000.0,
        math.radians(longitude_deg),
        math.radians(latitude_deg),
    ),
    dtype=float,
)

_, radii_km = spice.bodvrd("MOON", "RADII", 3)
a_km, b_km, c_km = (float(r) for r in radii_km)
nearest_point_km, observer_altitude_km = spice.nearpt(
    observer_xyz_km, a_km, b_km, c_km
)
ellipsoid_up = np.asarray(
    spice.surfnm(a_km, b_km, c_km, nearest_point_km),
    dtype=float,
)
ellipsoid_up /= np.linalg.norm(ellipsoid_up)

body_north_pole = np.array([0.0, 0.0, 1.0])
ellipsoid_north = body_north_pole - np.dot(body_north_pole, ellipsoid_up) * ellipsoid_up
north_norm = np.linalg.norm(ellipsoid_north)
if north_norm < 1e-10:
    raise ValueError("S04-P02 is too close to a pole to define north with this basis.")
ellipsoid_north /= north_norm
ellipsoid_east = np.cross(ellipsoid_north, ellipsoid_up)
ellipsoid_east /= np.linalg.norm(ellipsoid_east)

# Radial-sphere basis from the original run, reproduced for a direct model comparison.
lon = math.radians(longitude_deg)
lat = math.radians(latitude_deg)
sphere_east = np.array([-math.sin(lon), math.cos(lon), 0.0])
sphere_north = np.array([
    -math.sin(lat) * math.cos(lon),
    -math.sin(lat) * math.sin(lon),
    math.cos(lat),
])
sphere_up = observer_xyz_km / np.linalg.norm(observer_xyz_km)

assert abs(np.dot(ellipsoid_up, ellipsoid_north)) < 1e-12
assert abs(np.dot(ellipsoid_up, ellipsoid_east)) < 1e-12
assert abs(np.dot(ellipsoid_north, ellipsoid_east)) < 1e-12
assert np.dot(np.cross(ellipsoid_east, ellipsoid_north), ellipsoid_up) > 0.999999

print({
    "observer_altitude_above_pck_ellipsoid_km": float(observer_altitude_km),
    "moon_pck_radii_km": [a_km, b_km, c_km],
    "angle_radial_vs_ellipsoid_up_deg": math.degrees(
        math.acos(np.clip(np.dot(sphere_up, ellipsoid_up), -1.0, 1.0))
    ),
})

## 3. Compare vector conversion with NAIF azlcpo

We use the same target, epoch, observer position, frame, and `NONE` aberration correction for both routes. Our route gets the Moon-centred target position from `spkpos`, subtracts the site position, then projects into the locally constructed ellipsoid basis. The NAIF route calls `azlcpo` with `ELLIPSOID`.

The previous notebook's radial-sphere azimuth/elevation is included as a comparison column. Its difference from the ellipsoid-normal result is a change in local-vertical model, not the B4 software-conversion error.

In [ ]:
def angles_in_basis(vector_km, east_hat, north_hat, up_hat):
    east_component = float(np.dot(vector_km, east_hat))
    north_component = float(np.dot(vector_km, north_hat))
    up_component = float(np.dot(vector_km, up_hat))
    azimuth_deg = math.degrees(math.atan2(east_component, north_component)) % 360.0
    elevation_deg = math.degrees(
        math.atan2(up_component, math.hypot(east_component, north_component))
    )
    return azimuth_deg, elevation_deg

def circular_difference_deg(angle_a, angle_b):
    return (angle_a - angle_b + 180.0) % 360.0 - 180.0

times_utc = [
    "2025-01-01T00:00:00Z",
    "2025-01-01T12:00:00Z",
    "2025-01-01T23:00:00Z",
]
records = []

for utc in times_utc:
    for target in ("SUN", "EARTH"):
        match = baseline_df[
            (baseline_df["utc"] == utc) & (baseline_df["target"] == target)
        ]
        if len(match) != 1:
            raise ValueError(f"Expected one baseline row for {utc} / {target}, found {len(match)}")
        row = match.iloc[0]
        et = float(row["et_seconds_past_j2000_tdb"])

        target_from_moon_km, _ = spice.spkpos(
            target, et, "MOON_ME", "NONE", "MOON"
        )
        site_to_target_km = np.asarray(target_from_moon_km, dtype=float) - observer_xyz_km

        own_ellipsoid_az_deg, own_ellipsoid_el_deg = angles_in_basis(
            site_to_target_km, ellipsoid_east, ellipsoid_north, ellipsoid_up
        )
        own_sphere_az_deg, own_sphere_el_deg = angles_in_basis(
            site_to_target_km, sphere_east, sphere_north, sphere_up
        )

        azlsta, light_time_s = spice.azlcpo(
            "ELLIPSOID",
            target,
            et,
            "NONE",
            False,  # azimuth clockwise from local +X (north), toward east
            True,   # elevation positive toward local +Z (outward normal)
            observer_xyz_km,
            "MOON",
            "MOON_ME",
        )
        naif_az_deg = math.degrees(float(azlsta[1])) % 360.0
        naif_el_deg = math.degrees(float(azlsta[2]))

        az_delta = circular_difference_deg(own_ellipsoid_az_deg, naif_az_deg)
        el_delta = own_ellipsoid_el_deg - naif_el_deg
        records.append({
            "utc": utc,
            "target": target,
            "spice_frame": "MOON_ME",
            "aberration_correction": "NONE",
            "observer_x_km": float(observer_xyz_km[0]),
            "observer_y_km": float(observer_xyz_km[1]),
            "observer_z_km": float(observer_xyz_km[2]),
            "naif_azlcpo_azimuth_deg": naif_az_deg,
            "vector_ellipsoid_azimuth_deg": own_ellipsoid_az_deg,
            "azimuth_difference_deg_signed": az_delta,
            "naif_azlcpo_elevation_deg": naif_el_deg,
            "vector_ellipsoid_elevation_deg": own_ellipsoid_el_deg,
            "elevation_difference_deg_signed": el_delta,
            "baseline_radial_azimuth_deg": float(row["azimuth_deg_clockwise_from_north"]),
            "baseline_radial_elevation_deg": float(row["geometric_center_elevation_deg"]),
            "radial_vs_ellipsoid_azimuth_difference_deg": circular_difference_deg(
                own_sphere_az_deg, own_ellipsoid_az_deg
            ),
            "radial_vs_ellipsoid_elevation_difference_deg": own_sphere_el_deg - own_ellipsoid_el_deg,
            "light_time_s_returned": float(light_time_s),
        })

comparison = pd.DataFrame(records)
comparison["azimuth_abs_error_deg"] = comparison["azimuth_difference_deg_signed"].abs()
comparison["elevation_abs_error_deg"] = comparison["elevation_difference_deg_signed"].abs()
max_az_error = float(comparison["azimuth_abs_error_deg"].max())
max_el_error = float(comparison["elevation_abs_error_deg"].max())
b4_pass = max(max_az_error, max_el_error) < TOLERANCE_DEG

display(comparison)
print({
    "B4_pass": b4_pass,
    "proposed_tolerance_deg": TOLERANCE_DEG,
    "max_abs_azimuth_error_deg": max_az_error,
    "max_abs_elevation_error_deg": max_el_error,
})
if not b4_pass:
    print("B4 remains open. Inspect azimuth sense, local basis construction, and SPICE inputs.")

In [ ]:
comparison.to_csv(CROSSCHECK_CSV_PATH, index=False)

validation_manifest = {
    "run_utc": datetime.now(timezone.utc).isoformat(),
    "validation": "B4 local angle conversion cross-check",
    "method": "vector projection into locally constructed ellipsoid-normal basis vs SpiceyPy azlcpo ELLIPSOID",
    "baseline_manifest": str(BASELINE_MANIFEST_PATH),
    "baseline_csv": str(CSV_PATH),
    "point_id": point["id"],
    "reference_frame": "MOON_ME",
    "ephemeris": baseline.get("ephemeris"),
    "aberration_correction": "NONE",
    "epochs_checked_utc": times_utc,
    "targets": ["SUN", "EARTH"],
    "comparisons": int(len(comparison)),
    "tolerance_deg": TOLERANCE_DEG,
    "max_abs_azimuth_error_deg": max_az_error,
    "max_abs_elevation_error_deg": max_el_error,
    "B4_pass": bool(b4_pass),
    "kernel_manifest": baseline["kernel_manifest"],
    "notebook_wall_seconds": time.perf_counter() - NOTEBOOK_RUN_T0,
    "limits": [
        "Same DE421 kernels and ephemeris are used by both routes.",
        "The check validates local angle conversion, not independent ephemeris accuracy.",
        "Neither output models terrain occlusion or extended target disk fraction.",
        "Radial-sphere versus ellipsoid-normal differences are reported separately.",
    ],
}
CROSSCHECK_MANIFEST_PATH.write_text(
    json.dumps(validation_manifest, indent=2), encoding="utf-8"
)
print("saved_csv:", CROSSCHECK_CSV_PATH)
print("saved_manifest:", CROSSCHECK_MANIFEST_PATH)
print("validation wall seconds:", validation_manifest["notebook_wall_seconds"])

## Interpretation

A passing B4 result means the vector-to-local-angle calculation agrees with NAIF `azlcpo` for this observer, kernel set, six target/epoch combinations, ellipsoid normal, and the stated flags. It does not prove the ephemeris is independently correct, and it does not make the earlier radial-sphere angles terrain-aware. Keep the benchmark result scoped to this point and these epochs.

If B4 fails, do not change the tolerance to make it pass. First inspect the sign convention, axes, observer units, longitude/latitude, ellipsoid radii, and whether both routes use the same epoch and aberration setting.